<a href="https://colab.research.google.com/github/ahmedyomiisiaka/RP-Network-Analysis/blob/main/notebooks/06_Functional_Annotation_Enrichment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 06 — Functional Annotation and Enrichment Analysis

## Retinitis Pigmentosa Retinal Context-Specific Network

### Objective

This notebook investigates the biological functions, pathways, and functional
groups represented in the retinal context-specific RP protein interaction
network generated in Notebook 05.

The final Stage 5 network contains:

- 9,149 retinal context-specific interactions
- 3,748 unique proteins
- 105 RP seed proteins

The network was defined using:

- STRING combined score >= 0.400
- COMPARTMENTS score >= 2.0 with informative shared localisation
- Human Protein Atlas retina expression nTPM > 0 for both interacting proteins

IID and TissueNet were retained as additional supporting PPI evidence.

### Stage 6 aims

1. Prepare and quality-control enrichment input gene sets.
2. Annotate network proteins with biological functions.
3. Perform Gene Ontology enrichment.
4. Perform Reactome pathway enrichment.
5. Perform KEGG pathway enrichment.
6. Compare RP seed proteins and retinal-network interaction partners.
7. Identify functional groups/modules in which proteins work collectively.
8. Investigate pathways relevant to retinal degeneration and RP pathogenesis.
9. Prepare the network and functional context for subsequent PathLinker analysis.

In [2]:
# ================================================================
# NOTEBOOK 06 — FUNCTIONAL ANNOTATION AND ENRICHMENT ANALYSIS
# Stage 6 setup
# ================================================================

from google.colab import drive
from pathlib import Path
import pandas as pd
import numpy as np
import os

# Mount Google Drive
drive.mount("/content/drive")

# ------------------------------------------------
# Project paths
# ------------------------------------------------

PROJECT_DIR = Path("/content/drive/MyDrive/rp_network")

DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"

# Final Stage 5 retinal network
NETWORK_FILE = PROCESSED_DIR / "retinal_context_specific_network.tsv"

print("=" * 70)
print("STAGE 6 — FUNCTIONAL ANNOTATION AND ENRICHMENT")
print("=" * 70)

print("\nProject directory:")
print(PROJECT_DIR)

print("\nFinal retinal network:")
print(NETWORK_FILE)

print("\nFile exists:", NETWORK_FILE.exists())

if NETWORK_FILE.exists():
    print(f"File size: {NETWORK_FILE.stat().st_size / 1024**2:.2f} MB")
else:
    print("\nERROR: Final Stage 5 network was not found.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
STAGE 6 — FUNCTIONAL ANNOTATION AND ENRICHMENT

Project directory:
/content/drive/MyDrive/rp_network

Final retinal network:
/content/drive/MyDrive/rp_network/data/processed/retinal_context_specific_network.tsv

File exists: True
File size: 25.67 MB


In [3]:
# ================================================================
# 6.1 — LOAD AND QUALITY-CHECK FINAL STAGE 5 NETWORK
# ================================================================

print("=" * 70)
print("STAGE 6.1 — LOAD FINAL RETINAL CONTEXT-SPECIFIC NETWORK")
print("=" * 70)

# Load final Stage 5 network
network = pd.read_csv(NETWORK_FILE, sep="\t")

# ------------------------------------------------
# Basic integrity checks
# ------------------------------------------------

n_interactions = len(network)

all_proteins = set(network["protein_A"].dropna()) | \
               set(network["protein_B"].dropna())

n_proteins = len(all_proteins)

# RP seeds appearing on either side of an interaction
rp_A = set(
    network.loc[
        network["A_is_RP_seed"] == True,
        "protein_A"
    ].dropna()
)

rp_B = set(
    network.loc[
        network["B_is_RP_seed"] == True,
        "protein_B"
    ].dropna()
)

rp_seeds = rp_A | rp_B
n_rp_seeds = len(rp_seeds)

# Non-RP interaction partners
partner_proteins = all_proteins - rp_seeds
n_partners = len(partner_proteins)

# ------------------------------------------------
# Report
# ------------------------------------------------

print(f"\nInteractions loaded : {n_interactions:,}")
print(f"Unique proteins     : {n_proteins:,}")
print(f"RP seed proteins    : {n_rp_seeds:,}")
print(f"Non-RP partners     : {n_partners:,}")

print("\nSTRING score range:")
print(
    f"{network['string_score'].min():.3f} – "
    f"{network['string_score'].max():.3f}"
)

print("\nRetinal context status:")
print(network["retinal_context_pass"].value_counts(dropna=False))

# ------------------------------------------------
# Expected Stage 5 values
# ------------------------------------------------

print("\n" + "=" * 70)
print("EXPECTED STAGE 5 VALUES")
print("=" * 70)

checks = {
    "Interactions = 9,149": n_interactions == 9149,
    "Proteins = 3,748": n_proteins == 3748,
    "RP seeds = 105": n_rp_seeds == 105,
    "All rows pass retinal context":
        network["retinal_context_pass"].fillna(False).all()
}

for description, passed in checks.items():
    status = "PASS" if passed else "FAIL"
    print(f"{description:<40} [{status}]")

print("\n" + "=" * 70)

if all(checks.values()):
    print("STAGE 6 INPUT QC: PASS")
    print("The final Stage 5 retinal network is ready for enrichment analysis.")
else:
    print("STAGE 6 INPUT QC: REVIEW REQUIRED")
    print("Do not proceed until failed checks are investigated.")

print("=" * 70)

STAGE 6.1 — LOAD FINAL RETINAL CONTEXT-SPECIFIC NETWORK

Interactions loaded : 9,149
Unique proteins     : 3,748
RP seed proteins    : 105
Non-RP partners     : 3,643

STRING score range:
0.400 – 0.999

Retinal context status:
retinal_context_pass
True    9149
Name: count, dtype: int64

EXPECTED STAGE 5 VALUES
Interactions = 9,149                     [PASS]
Proteins = 3,748                         [PASS]
RP seeds = 105                           [PASS]
All rows pass retinal context            [PASS]

STAGE 6 INPUT QC: PASS
The final Stage 5 retinal network is ready for enrichment analysis.


In [4]:
# ================================================================
# 6.2 — PREPARE ENRICHMENT INPUT GENE SETS
# ================================================================

from pathlib import Path
import pandas as pd

print("=" * 70)
print("STAGE 6.2 — PREPARE ENRICHMENT INPUT GENE SETS")
print("=" * 70)

# ------------------------------------------------
# Create output directory
# ------------------------------------------------

ENRICHMENT_INPUT_DIR = (
    PROCESSED_DIR / "enrichment_inputs"
)

ENRICHMENT_INPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------
# Clean and sort the three gene sets
# ------------------------------------------------

rp_seed_genes = sorted(
    str(gene).strip().upper()
    for gene in rp_seeds
    if pd.notna(gene)
)

partner_genes = sorted(
    str(gene).strip().upper()
    for gene in partner_proteins
    if pd.notna(gene)
)

all_retinal_genes = sorted(
    str(gene).strip().upper()
    for gene in all_proteins
    if pd.notna(gene)
)

# ------------------------------------------------
# QC
# ------------------------------------------------

print("\nGene set sizes:")
print(f"RP seeds                 : {len(rp_seed_genes):,}")
print(f"Non-RP network partners  : {len(partner_genes):,}")
print(f"All retinal proteins     : {len(all_retinal_genes):,}")

assert len(rp_seed_genes) == 105
assert len(partner_genes) == 3643
assert len(all_retinal_genes) == 3748

# Check that RP seeds and partners do not overlap
overlap = set(rp_seed_genes) & set(partner_genes)

print(f"\nRP seed / partner overlap: {len(overlap)}")

assert len(overlap) == 0

# Check union
combined = set(rp_seed_genes) | set(partner_genes)

assert combined == set(all_retinal_genes)

print("✓ RP seeds + partners reproduce the complete 3,748-protein network.")

# ------------------------------------------------
# Save helper function
# ------------------------------------------------

def save_gene_set(genes, basename):

    # Plain text — one gene per line
    txt_file = ENRICHMENT_INPUT_DIR / f"{basename}.txt"

    with open(txt_file, "w", encoding="utf-8") as f:
        f.write("\n".join(genes) + "\n")

    # TSV — useful for documentation/data processing
    tsv_file = ENRICHMENT_INPUT_DIR / f"{basename}.tsv"

    pd.DataFrame({
        "gene_symbol": genes
    }).to_csv(
        tsv_file,
        sep="\t",
        index=False
    )

    return txt_file, tsv_file


# ------------------------------------------------
# Save all three sets
# ------------------------------------------------

rp_txt, rp_tsv = save_gene_set(
    rp_seed_genes,
    "rp_seeds_105"
)

partners_txt, partners_tsv = save_gene_set(
    partner_genes,
    "retinal_network_partners_3643"
)

all_txt, all_tsv = save_gene_set(
    all_retinal_genes,
    "all_retinal_network_proteins_3748"
)

# ------------------------------------------------
# Report
# ------------------------------------------------

print("\n" + "=" * 70)
print("FILES CREATED")
print("=" * 70)

for path in [
    rp_txt,
    rp_tsv,
    partners_txt,
    partners_tsv,
    all_txt,
    all_tsv
]:
    print("✓", path.name)

# ------------------------------------------------
# Display the complete RP seed list
# ------------------------------------------------

print("\n" + "=" * 70)
print("105 RP SEED GENES")
print("=" * 70)

for i, gene in enumerate(rp_seed_genes, start=1):
    print(f"{i:3}. {gene}")

print("\n" + "=" * 70)
print("STAGE 6.2 COMPLETE")
print("=" * 70)

STAGE 6.2 — PREPARE ENRICHMENT INPUT GENE SETS

Gene set sizes:
RP seeds                 : 105
Non-RP network partners  : 3,643
All retinal proteins     : 3,748

RP seed / partner overlap: 0
✓ RP seeds + partners reproduce the complete 3,748-protein network.

FILES CREATED
✓ rp_seeds_105.txt
✓ rp_seeds_105.tsv
✓ retinal_network_partners_3643.txt
✓ retinal_network_partners_3643.tsv
✓ all_retinal_network_proteins_3748.txt
✓ all_retinal_network_proteins_3748.tsv

105 RP SEED GENES
  1. AGBL5
  2. AHI1
  3. AIPL1
  4. ARHGEF18
  5. ARL2BP
  6. ARL3
  7. ARL6
  8. BBS1
  9. BBS2
 10. BBS9
 11. BCOR
 12. BEST1
 13. C19ORF44
 14. CC2D2A
 15. CDHR1
 16. CEP162
 17. CEP250
 18. CERKL
 19. CFAP20
 20. CFAP410
 21. CLCC1
 22. CLN3
 23. CLRN1
 24. CNGA1
 25. CNGB1
 26. COQ2
 27. COQ5
 28. COQ8B
 29. CRB1
 30. CREB3
 31. CRX
 32. CYP4V2
 33. DHDDS
 34. DHX38
 35. DYNC2H1
 36. EYS
 37. FAM161A
 38. FLVCR1
 39. FSD1L
 40. GNAT1
 41. GUCA1A
 42. GUCY2D
 43. HGSNAT
 44. HK1
 45. IDH3A
 46. IDH3B
 47. I

In [5]:
# ================================================================
# 6.3b — LOCATE AND QC PANTHER GO BIOLOGICAL PROCESS RESULT
# ================================================================

from pathlib import Path
import pandas as pd

print("=" * 70)
print("STAGE 6.3b — PANTHER GO BIOLOGICAL PROCESS RESULT QC")
print("=" * 70)

filename = "PANTHER_GO_BP_RP_seeds_raw.txt"

# Search inside the RP project on Google Drive
search_root = Path("/content/drive/MyDrive/rp_network")

matches = list(search_root.rglob(filename))

print(f"\nSearching for:\n{filename}")

if len(matches) == 0:
    print("\n✗ FILE NOT FOUND")
    print("Upload/copy the renamed PANTHER file into:")
    print("/content/drive/MyDrive/rp_network/")

elif len(matches) > 1:
    print(f"\n⚠ Found {len(matches)} copies:")
    for p in matches:
        print(" -", p)

else:
    panther_file = matches[0]

    print("\n✓ FILE FOUND")
    print(panther_file)

    # ------------------------------------------------------------
    # Read PANTHER table
    # Metadata occupies the first 11 lines.
    # Line 12 contains the table header.
    # ------------------------------------------------------------

    panther_bp = pd.read_csv(
        panther_file,
        sep="\t",
        skiprows=11
    )

    print("\n" + "=" * 70)
    print("TABLE LOADED")
    print("=" * 70)

    print(f"Rows    : {len(panther_bp):,}")
    print(f"Columns : {len(panther_bp.columns)}")

    print("\nColumn names:")
    for col in panther_bp.columns:
        print(" -", col)

    print("\nFirst 10 rows:")
    display(panther_bp.head(10))

STAGE 6.3b — PANTHER GO BIOLOGICAL PROCESS RESULT QC

Searching for:
PANTHER_GO_BP_RP_seeds_raw.txt

✓ FILE FOUND
/content/drive/MyDrive/rp_network/data/processed/enrichment_inputs/PANTHER_GO_BP_RP_seeds_raw.txt

TABLE LOADED
Rows    : 95
Columns : 8

Column names:
 - GO biological process complete
 - Homo sapiens - REFLIST (20580)
 - rp_seeds_105.txt (109)
 - rp_seeds_105.txt (expected)
 - rp_seeds_105.txt (over/under)
 - rp_seeds_105.txt (fold Enrichment)
 - rp_seeds_105.txt (raw P-value)
 - rp_seeds_105.txt (FDR)

First 10 rows:


,GO biological process complete,Homo sapiens - REFLIST (20580),rp_seeds_105.txt (109),rp_seeds_105.txt (expected),rp_seeds_105.txt (over/under),rp_seeds_105.txt (fold Enrichment),rp_seeds_105.txt (raw P-value),rp_seeds_105.txt (FDR)
0,transducin-mediated opsin signaling pathway (G...,4,4,0.02,+,> 100,7.450000e-10,3.420000e-07
1,photoreceptor cell outer segment organization ...,7,6,0.04,+,> 100,1.340000e-13,8.770000e-11
2,G protein-coupled opsin signaling pathway (GO:...,6,5,0.03,+,> 100,2.270000e-11,1.200000e-08
3,protein localization to photoreceptor outer se...,3,2,0.02,+,> 100,8.310000e-05,1.360000e-02
4,"phototransduction, visible light (GO:0007603)",11,6,0.06,+,> 100,8.680000e-12,4.780000e-09
5,isocitrate metabolic process (GO:0006102),4,2,0.02,+,94.40,1.660000e-04,2.530000e-02
6,photoreceptor cell morphogenesis (GO:0008594),4,2,0.02,+,94.40,1.660000e-04,2.500000e-02
7,detection of light stimulus involved in sensor...,17,8,0.09,+,88.85,1.110000e-14,9.580000e-12
8,detection of light stimulus involved in visual...,17,8,0.09,+,88.85,1.110000e-14,9.020000e-12
9,vitamin A metabolic process (GO:0006776),7,3,0.04,+,80.92,4.980000e-06,1.140000e-03


In [6]:
# ================================================================
# 6.3c — CLEAN AND SUMMARIZE PANTHER GO BP RESULTS
# ================================================================

import pandas as pd
import re

print("=" * 70)
print("STAGE 6.3c — CLEAN PANTHER GO BIOLOGICAL PROCESS RESULTS")
print("=" * 70)

# ------------------------------------------------
# Make a clean working copy
# ------------------------------------------------

go_bp = panther_bp.copy()

# Rename columns
go_bp.columns = [
    "GO_term_raw",
    "reference_count",
    "observed_count",
    "expected_count",
    "direction",
    "fold_enrichment_raw",
    "raw_p_value",
    "FDR"
]

# ------------------------------------------------
# Extract GO ID and term name
# ------------------------------------------------

go_bp["GO_ID"] = go_bp["GO_term_raw"].str.extract(
    r"\((GO:\d+)\)"
)

go_bp["GO_term"] = (
    go_bp["GO_term_raw"]
    .str.replace(r"\s*\(GO:\d+\)\s*$", "", regex=True)
    .str.strip()
)

# ------------------------------------------------
# Convert numeric columns
# ------------------------------------------------

for col in [
    "reference_count",
    "observed_count",
    "expected_count",
    "raw_p_value",
    "FDR"
]:
    go_bp[col] = pd.to_numeric(go_bp[col], errors="coerce")

# Preserve original fold-enrichment text
go_bp["fold_enrichment_raw"] = (
    go_bp["fold_enrichment_raw"]
    .astype(str)
    .str.strip()
)

# Numeric representation:
# PANTHER reports some values as "> 100".
# We use 100 numerically only for sorting/plotting,
# while retaining the original "> 100" field.
go_bp["fold_enrichment_numeric"] = (
    go_bp["fold_enrichment_raw"]
    .str.replace(">", "", regex=False)
    .str.strip()
)

go_bp["fold_enrichment_numeric"] = pd.to_numeric(
    go_bp["fold_enrichment_numeric"],
    errors="coerce"
)

# ------------------------------------------------
# Significance
# ------------------------------------------------

go_bp["significant_FDR_0.05"] = go_bp["FDR"] < 0.05

significant = go_bp[
    go_bp["significant_FDR_0.05"]
].copy()

overrepresented = significant[
    significant["direction"] == "+"
].copy()

underrepresented = significant[
    significant["direction"] == "-"
].copy()

# ------------------------------------------------
# QC summary
# ------------------------------------------------

print("\nTotal PANTHER result rows:")
print(f"{len(go_bp):,}")

print("\nSignificant at FDR < 0.05:")
print(f"{len(significant):,}")

print("\nSignificant overrepresented (+):")
print(f"{len(overrepresented):,}")

print("\nSignificant underrepresented (-):")
print(f"{len(underrepresented):,}")

print("\nRows without a standard GO ID:")
print(f"{go_bp['GO_ID'].isna().sum():,}")

# ------------------------------------------------
# Top 20 significant overrepresented processes
# Sort by FDR first
# ------------------------------------------------

top20 = (
    overrepresented
    .sort_values(
        ["FDR", "fold_enrichment_numeric"],
        ascending=[True, False]
    )
    .head(20)
)

print("\n" + "=" * 70)
print("TOP 20 SIGNIFICANT OVERREPRESENTED GO BIOLOGICAL PROCESSES")
print("=" * 70)

display(
    top20[
        [
            "GO_ID",
            "GO_term",
            "observed_count",
            "expected_count",
            "fold_enrichment_raw",
            "raw_p_value",
            "FDR"
        ]
    ]
)

# ------------------------------------------------
# Show underrepresented processes
# ------------------------------------------------

print("\n" + "=" * 70)
print("SIGNIFICANT UNDERREPRESENTED PROCESSES")
print("=" * 70)

display(
    underrepresented[
        [
            "GO_ID",
            "GO_term",
            "observed_count",
            "expected_count",
            "fold_enrichment_raw",
            "FDR"
        ]
    ].sort_values("FDR")
)

# ------------------------------------------------
# Save clean tables
# ------------------------------------------------

output_dir = (
    Path("/content/drive/MyDrive/rp_network/"
         "data/processed/enrichment_results")
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

all_output = output_dir / "PANTHER_GO_BP_RP_seeds_clean.tsv"

sig_output = (
    output_dir /
    "PANTHER_GO_BP_RP_seeds_significant_FDR05.tsv"
)

go_bp.to_csv(
    all_output,
    sep="\t",
    index=False
)

significant.to_csv(
    sig_output,
    sep="\t",
    index=False
)

print("\n" + "=" * 70)
print("FILES SAVED")
print("=" * 70)

print("✓", all_output)
print("✓", sig_output)

print("\nSTAGE 6.3c COMPLETE")

STAGE 6.3c — CLEAN PANTHER GO BIOLOGICAL PROCESS RESULTS

Total PANTHER result rows:
95

Significant at FDR < 0.05:
95

Significant overrepresented (+):
93

Significant underrepresented (-):
2

Rows without a standard GO ID:
1

TOP 20 SIGNIFICANT OVERREPRESENTED GO BIOLOGICAL PROCESSES


,GO_ID,GO_term,observed_count,expected_count,fold_enrichment_raw,raw_p_value,FDR
21,GO:0050953,sensory perception of light stimulus,33,0.81,40.99,6.280000e-45,8.640000e-41
22,GO:0007601,visual perception,31,0.77,40.09,8.000000e-42,5.500000e-38
64,GO:0007600,sensory perception,35,4.49,7.80,3.750000e-22,1.720000e-18
13,GO:0001895,retina homeostasis,14,0.24,58.74,8.380000e-22,2.880000e-18
47,GO:0060271,cilium assembly,24,1.75,13.69,9.360000e-21,2.150000e-17
17,GO:0009583,detection of light stimulus,14,0.28,50.83,8.620000e-21,2.370000e-17
49,GO:0044782,cilium organization,24,1.94,12.35,1.050000e-19,2.070000e-16
12,GO:0045494,photoreceptor cell maintenance,12,0.19,62.94,2.940000e-19,5.060000e-16
55,GO:0120031,plasma membrane bounded cell projection assembly,24,2.12,11.33,7.750000e-19,1.190000e-15
69,GO:0050877,nervous system process,36,6.14,5.86,1.010000e-18,1.260000e-15



SIGNIFICANT UNDERREPRESENTED PROCESSES


,GO_ID,GO_term,observed_count,expected_count,fold_enrichment_raw,FDR
94,NaN,Unclassified (UNCLASSIFIED),3,16.43,.18,0.00748
93,GO:0006950,response to stress,4,16.07,.25,0.04910



FILES SAVED
✓ /content/drive/MyDrive/rp_network/data/processed/enrichment_results/PANTHER_GO_BP_RP_seeds_clean.tsv
✓ /content/drive/MyDrive/rp_network/data/processed/enrichment_results/PANTHER_GO_BP_RP_seeds_significant_FDR05.tsv

STAGE 6.3c COMPLETE


In [8]:
# ================================================================
# STAGE 6.4b — PANTHER GO BIOLOGICAL PROCESS
# RETINAL NETWORK PARTNERS — RESULT QC
# CORRECTED FOR PANTHER METADATA HEADER
# ================================================================

from pathlib import Path
import pandas as pd

print("=" * 70)
print("STAGE 6.4b — PANTHER GO BP RETINAL PARTNERS RESULT QC")
print("=" * 70)

input_dir = Path(
    "/content/drive/MyDrive/rp_network/data/processed/enrichment_inputs"
)

panther_file = input_dir / "PANTHER_GO_BP_retinal_partners_raw.txt"

print("\nSearching for:")
print(panther_file.name)

if not panther_file.exists():
    raise FileNotFoundError(
        f"FILE NOT FOUND:\n{panther_file}"
    )

print("\n✓ FILE FOUND")
print(panther_file)

# ------------------------------------------------
# PANTHER export structure:
# lines 1–11  = metadata
# line 12     = actual table header
# ------------------------------------------------

df_partners_raw = pd.read_csv(
    panther_file,
    sep="\t",
    skiprows=11,
    low_memory=False
)

print("\n" + "=" * 70)
print("TABLE LOADED")
print("=" * 70)

print(f"Rows    : {len(df_partners_raw):,}")
print(f"Columns : {len(df_partners_raw.columns)}")

print("\nColumn names:")
for col in df_partners_raw.columns:
    print(" -", col)

print("\nFirst 10 rows:")
display(df_partners_raw.head(10))

print("\n" + "=" * 70)
print("BASIC QC")
print("=" * 70)

print("\nMissing values by column:")
print(df_partners_raw.isna().sum())

print("\nDuplicate complete rows:")
print(df_partners_raw.duplicated().sum())

print("\n" + "=" * 70)
print("STAGE 6.4b QC LOAD COMPLETE")
print("=" * 70)

STAGE 6.4b — PANTHER GO BP RETINAL PARTNERS RESULT QC

Searching for:
PANTHER_GO_BP_retinal_partners_raw.txt

✓ FILE FOUND
/content/drive/MyDrive/rp_network/data/processed/enrichment_inputs/PANTHER_GO_BP_retinal_partners_raw.txt

TABLE LOADED
Rows    : 2,105
Columns : 8

Column names:
 - GO biological process complete
 - Homo sapiens - REFLIST (20580)
 - retinal_network_partners_3643.txt (3709)
 - retinal_network_partners_3643.txt (expected)
 - retinal_network_partners_3643.txt (over/under)
 - retinal_network_partners_3643.txt (fold Enrichment)
 - retinal_network_partners_3643.txt (raw P-value)
 - retinal_network_partners_3643.txt (FDR)

First 10 rows:


,GO biological process complete,Homo sapiens - REFLIST (20580),retinal_network_partners_3643.txt (3709),retinal_network_partners_3643.txt (expected),retinal_network_partners_3643.txt (over/under),retinal_network_partners_3643.txt (fold Enrichment),retinal_network_partners_3643.txt (raw P-value),retinal_network_partners_3643.txt (FDR)
0,protein transport to vacuole involved in ubiqu...,15,15,2.70,+,5.55,6.720000e-12,2.530000e-10
1,U2-type prespliceosome assembly (GO:1903241),25,25,4.51,+,5.55,2.320000e-19,1.850000e-17
2,activation of protein kinase A activity (GO:00...,5,5,0.90,+,5.55,1.900000e-04,2.140000e-03
3,hypoxanthine biosynthetic process (GO:0046101),3,3,0.54,+,5.55,5.850000e-03,4.090000e-02
4,dolichol phosphate mannose biosynthetic proces...,3,3,0.54,+,5.55,5.850000e-03,4.090000e-02
5,central nervous system maturation (GO:0021626),3,3,0.54,+,5.55,5.850000e-03,4.090000e-02
6,cell adhesion molecule production (GO:0060352),3,3,0.54,+,5.55,5.850000e-03,4.080000e-02
7,TMP catabolic process (GO:0046045),3,3,0.54,+,5.55,5.850000e-03,4.080000e-02
8,TMP metabolic process (GO:0046044),3,3,0.54,+,5.55,5.850000e-03,4.080000e-02
9,regulation of ryanodine-sensitive calcium-rele...,5,5,0.90,+,5.55,1.900000e-04,2.140000e-03



BASIC QC

Missing values by column:
GO biological process complete                         0
Homo sapiens - REFLIST (20580)                         0
retinal_network_partners_3643.txt (3709)               0
retinal_network_partners_3643.txt (expected)           0
retinal_network_partners_3643.txt (over/under)         0
retinal_network_partners_3643.txt (fold Enrichment)    0
retinal_network_partners_3643.txt (raw P-value)        0
retinal_network_partners_3643.txt (FDR)                0
dtype: int64

Duplicate complete rows:
0

STAGE 6.4b QC LOAD COMPLETE


In [9]:
# ================================================================
# STAGE 6.4c — CLEAN AND SUMMARIZE PANTHER GO BP RESULTS
# RETINAL NETWORK PARTNERS
# ================================================================

import pandas as pd

print("=" * 70)
print("STAGE 6.4c — CLEAN GO BP RESULTS FOR RETINAL NETWORK PARTNERS")
print("=" * 70)

# ------------------------------------------------
# Working copy
# ------------------------------------------------

go_bp_partners = df_partners_raw.copy()

# Rename columns
go_bp_partners.columns = [
    "GO_term_raw",
    "reference_count",
    "observed_count",
    "expected_count",
    "direction",
    "fold_enrichment_raw",
    "raw_p_value",
    "FDR"
]

# ------------------------------------------------
# Extract GO ID and clean term name
# ------------------------------------------------

go_bp_partners["GO_ID"] = go_bp_partners["GO_term_raw"].str.extract(
    r"\((GO:\d+)\)"
)

go_bp_partners["GO_term"] = (
    go_bp_partners["GO_term_raw"]
    .str.replace(r"\s*\(GO:\d+\)\s*$", "", regex=True)
    .str.strip()
)

# ------------------------------------------------
# Convert numeric columns
# ------------------------------------------------

for col in [
    "reference_count",
    "observed_count",
    "expected_count",
    "raw_p_value",
    "FDR"
]:
    go_bp_partners[col] = pd.to_numeric(
        go_bp_partners[col],
        errors="coerce"
    )

# Keep original fold-enrichment text
go_bp_partners["fold_enrichment_raw"] = (
    go_bp_partners["fold_enrichment_raw"]
    .astype(str)
    .str.strip()
)

# Numeric fold enrichment for sorting
go_bp_partners["fold_enrichment_numeric"] = (
    go_bp_partners["fold_enrichment_raw"]
    .str.replace(">", "", regex=False)
    .str.strip()
)

go_bp_partners["fold_enrichment_numeric"] = pd.to_numeric(
    go_bp_partners["fold_enrichment_numeric"],
    errors="coerce"
)

# ------------------------------------------------
# Significance
# ------------------------------------------------

go_bp_partners["significant_FDR_0.05"] = (
    go_bp_partners["FDR"] < 0.05
)

significant_partners = go_bp_partners[
    go_bp_partners["significant_FDR_0.05"]
].copy()

overrepresented_partners = significant_partners[
    significant_partners["direction"] == "+"
].copy()

underrepresented_partners = significant_partners[
    significant_partners["direction"] == "-"
].copy()

# ------------------------------------------------
# Summary
# ------------------------------------------------

print("\nTotal PANTHER result rows:")
print(f"{len(go_bp_partners):,}")

print("\nSignificant at FDR < 0.05:")
print(f"{len(significant_partners):,}")

print("\nSignificant overrepresented (+):")
print(f"{len(overrepresented_partners):,}")

print("\nSignificant underrepresented (-):")
print(f"{len(underrepresented_partners):,}")

print("\nRows without a standard GO ID:")
print(f"{go_bp_partners['GO_ID'].isna().sum():,}")

# ------------------------------------------------
# Top 20 by FDR
# ------------------------------------------------

top20_partners = (
    overrepresented_partners
    .sort_values(
        ["FDR", "fold_enrichment_numeric"],
        ascending=[True, False]
    )
    .head(20)
)

print("\n" + "=" * 70)
print("TOP 20 SIGNIFICANT OVERREPRESENTED GO BP TERMS")
print("=" * 70)

display(
    top20_partners[
        [
            "GO_ID",
            "GO_term",
            "observed_count",
            "expected_count",
            "fold_enrichment_raw",
            "raw_p_value",
            "FDR"
        ]
    ]
)

# ------------------------------------------------
# Top 20 strongest fold-enrichment terms
# Require at least 3 observed proteins
# ------------------------------------------------

top_fold_partners = (
    overrepresented_partners[
        overrepresented_partners["observed_count"] >= 3
    ]
    .sort_values(
        ["fold_enrichment_numeric", "FDR"],
        ascending=[False, True]
    )
    .head(20)
)

print("\n" + "=" * 70)
print("TOP 20 BY FOLD ENRICHMENT (>=3 OBSERVED PROTEINS)")
print("=" * 70)

display(
    top_fold_partners[
        [
            "GO_ID",
            "GO_term",
            "observed_count",
            "expected_count",
            "fold_enrichment_raw",
            "FDR"
        ]
    ]
)

# ------------------------------------------------
# Underrepresented terms
# ------------------------------------------------

print("\n" + "=" * 70)
print("SIGNIFICANT UNDERREPRESENTED PROCESSES")
print("=" * 70)

display(
    underrepresented_partners[
        [
            "GO_ID",
            "GO_term",
            "observed_count",
            "expected_count",
            "fold_enrichment_raw",
            "FDR"
        ]
    ]
    .sort_values("FDR")
    .head(30)
)

# ------------------------------------------------
# Save clean tables
# ------------------------------------------------

output_dir = Path(
    "/content/drive/MyDrive/rp_network/"
    "data/processed/enrichment_results"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

all_output = (
    output_dir /
    "PANTHER_GO_BP_retinal_partners_clean.tsv"
)

sig_output = (
    output_dir /
    "PANTHER_GO_BP_retinal_partners_significant_FDR05.tsv"
)

go_bp_partners.to_csv(
    all_output,
    sep="\t",
    index=False
)

significant_partners.to_csv(
    sig_output,
    sep="\t",
    index=False
)

print("\n" + "=" * 70)
print("FILES SAVED")
print("=" * 70)

print("✓", all_output)
print("✓", sig_output)

print("\nSTAGE 6.4c COMPLETE")

STAGE 6.4c — CLEAN GO BP RESULTS FOR RETINAL NETWORK PARTNERS

Total PANTHER result rows:
2,105

Significant at FDR < 0.05:
2,105

Significant overrepresented (+):
2,080

Significant underrepresented (-):
25

Rows without a standard GO ID:
1

TOP 20 SIGNIFICANT OVERREPRESENTED GO BP TERMS


,GO_ID,GO_term,observed_count,expected_count,fold_enrichment_raw,raw_p_value,FDR
2046,GO:0009987,cellular process,3357,2692.00,1.25,1.470000e-189,2.030000e-185
1790,GO:0008152,metabolic process,1998,1275.98,1.57,5.800000e-160,3.990000e-156
1765,GO:0044238,primary metabolic process,1828,1147.66,1.59,3.590000e-148,1.650000e-144
1466,GO:0009058,biosynthetic process,1220,658.00,1.85,1.410000e-138,4.860000e-135
2070,GO:0008150,biological_process,3561,3149.77,1.13,1.810000e-122,4.140000e-119
1468,GO:0044085,cellular component biogenesis,921,497.78,1.85,3.550000e-98,6.980000e-95
1825,GO:0071840,cellular component organization or biogenesis,1523,992.85,1.53,7.470000e-98,1.280000e-94
302,GO:0008380,RNA splicing,222,55.69,3.99,5.900000e-97,9.020000e-94
261,GO:0000375,"RNA splicing, via transesterification reactions",200,47.94,4.17,1.950000e-93,2.690000e-90
263,GO:0000377,"RNA splicing, via transesterification reaction...",196,47.04,4.17,2.120000e-91,2.430000e-88



TOP 20 BY FOLD ENRICHMENT (>=3 OBSERVED PROTEINS)


,GO_ID,GO_term,observed_count,expected_count,fold_enrichment_raw,FDR
1,GO:1903241,U2-type prespliceosome assembly,25,4.51,5.55,1.850000e-17
0,GO:0043328,protein transport to vacuole involved in ubiqu...,15,2.70,5.55,2.530000e-10
59,GO:1904869,regulation of protein localization to Cajal body,10,1.80,5.55,8.380000e-07
58,GO:1904871,positive regulation of protein localization to...,10,1.80,5.55,8.390000e-07
13,GO:0006784,heme A biosynthetic process,9,1.62,5.55,4.110000e-06
34,GO:0006228,UTP biosynthetic process,9,1.62,5.55,4.110000e-06
12,GO:0006785,heme B biosynthetic process,9,1.62,5.55,4.120000e-06
81,GO:0016102,diterpenoid biosynthetic process,8,1.44,5.55,2.020000e-05
64,GO:0035735,intraciliary transport involved in cilium asse...,7,1.26,5.55,9.680000e-05
22,GO:0044784,metaphase/anaphase transition of cell cycle,7,1.26,5.55,9.690000e-05



SIGNIFICANT UNDERREPRESENTED PROCESSES


,GO_ID,GO_term,observed_count,expected_count,fold_enrichment_raw,FDR
2093,NaN,Unclassified (UNCLASSIFIED),148,559.23,.26,4.970000e-119
2099,GO:0050911,detection of chemical stimulus involved in sen...,3,78.76,.04,2.770000e-31
2098,GO:0050907,detection of chemical stimulus involved in sen...,8,87.23,.09,5.520000e-29
2097,GO:0007608,sensory perception of smell,12,82.18,.15,8.760000e-23
2095,GO:0007606,sensory perception of chemical stimulus,20,96.06,.21,5.410000e-22
2096,GO:0009593,detection of chemical stimulus,19,93.36,.20,1.230000e-21
2092,GO:0050906,detection of stimulus involved in sensory perc...,27,97.50,.28,9.540000e-18
2082,GO:0051606,detection of stimulus,76,118.23,.64,8.420000e-05
2086,GO:0002250,adaptive immune response,29,55.87,.52,3.740000e-04
2091,GO:0019724,B cell mediated immunity,9,26.85,.34,4.990000e-04



FILES SAVED
✓ /content/drive/MyDrive/rp_network/data/processed/enrichment_results/PANTHER_GO_BP_retinal_partners_clean.tsv
✓ /content/drive/MyDrive/rp_network/data/processed/enrichment_results/PANTHER_GO_BP_retinal_partners_significant_FDR05.tsv

STAGE 6.4c COMPLETE


In [10]:
# ================================================================
# STAGE 6.5 — COMPARE GO BIOLOGICAL PROCESS ENRICHMENT
# RP SEEDS vs RETINAL NETWORK PARTNERS
# ================================================================

from pathlib import Path
import pandas as pd

print("=" * 70)
print("STAGE 6.5 — RP SEEDS vs RETINAL PARTNERS GO-BP COMPARISON")
print("=" * 70)

results_dir = Path(
    "/content/drive/MyDrive/rp_network/data/processed/enrichment_results"
)

rp_file = results_dir / "PANTHER_GO_BP_RP_seeds_significant_FDR05.tsv"
partner_file = results_dir / "PANTHER_GO_BP_retinal_partners_significant_FDR05.tsv"

# ------------------------------------------------
# 1. Confirm files
# ------------------------------------------------

print("\nChecking input files:")

for f in [rp_file, partner_file]:
    print(f"{f.name}: {f.exists()}")

if not rp_file.exists() or not partner_file.exists():
    raise FileNotFoundError("One or more enrichment result files are missing.")

# ------------------------------------------------
# 2. Load
# ------------------------------------------------

rp = pd.read_csv(rp_file, sep="\t")
partners = pd.read_csv(partner_file, sep="\t")

print("\nRP significant terms      :", f"{len(rp):,}")
print("Partner significant terms :", f"{len(partners):,}")

# Remove PANTHER unclassified rows for GO-ID comparison
rp_go = rp.dropna(subset=["GO_ID"]).copy()
partner_go = partners.dropna(subset=["GO_ID"]).copy()

rp_ids = set(rp_go["GO_ID"])
partner_ids = set(partner_go["GO_ID"])

# ------------------------------------------------
# 3. Compare GO terms
# ------------------------------------------------

shared_ids = rp_ids & partner_ids
rp_only_ids = rp_ids - partner_ids
partner_only_ids = partner_ids - rp_ids

print("\n" + "=" * 70)
print("GO TERM OVERLAP")
print("=" * 70)

print(f"RP GO terms with IDs       : {len(rp_ids):,}")
print(f"Partner GO terms with IDs  : {len(partner_ids):,}")
print(f"Shared significant terms   : {len(shared_ids):,}")
print(f"RP-only significant terms  : {len(rp_only_ids):,}")
print(f"Partner-only terms          : {len(partner_only_ids):,}")

# ------------------------------------------------
# 4. Shared significant terms
# ------------------------------------------------

shared = (
    rp_go[rp_go["GO_ID"].isin(shared_ids)]
    [
        [
            "GO_ID",
            "GO_term",
            "observed_count",
            "expected_count",
            "fold_enrichment_raw",
            "FDR"
        ]
    ]
    .rename(
        columns={
            "observed_count": "RP_observed",
            "expected_count": "RP_expected",
            "fold_enrichment_raw": "RP_fold",
            "FDR": "RP_FDR"
        }
    )
)

partner_shared = (
    partner_go[partner_go["GO_ID"].isin(shared_ids)]
    [
        [
            "GO_ID",
            "observed_count",
            "expected_count",
            "fold_enrichment_raw",
            "FDR"
        ]
    ]
    .rename(
        columns={
            "observed_count": "partner_observed",
            "expected_count": "partner_expected",
            "fold_enrichment_raw": "partner_fold",
            "FDR": "partner_FDR"
        }
    )
)

shared = shared.merge(
    partner_shared,
    on="GO_ID",
    how="inner"
)

shared = shared.sort_values("RP_FDR")

print("\n" + "=" * 70)
print("TOP SHARED GO BIOLOGICAL PROCESSES")
print("=" * 70)

display(shared.head(30))

# ------------------------------------------------
# 5. RP-only terms
# ------------------------------------------------

rp_only = (
    rp_go[rp_go["GO_ID"].isin(rp_only_ids)]
    .sort_values("FDR")
)

print("\n" + "=" * 70)
print("TOP RP-SEED-SPECIFIC SIGNIFICANT PROCESSES")
print("=" * 70)

display(
    rp_only[
        [
            "GO_ID",
            "GO_term",
            "observed_count",
            "fold_enrichment_raw",
            "FDR"
        ]
    ].head(20)
)

# ------------------------------------------------
# 6. Partner-only terms
# ------------------------------------------------

partner_only = (
    partner_go[partner_go["GO_ID"].isin(partner_only_ids)]
    .sort_values("FDR")
)

print("\n" + "=" * 70)
print("TOP NETWORK-PARTNER-ONLY SIGNIFICANT PROCESSES")
print("=" * 70)

display(
    partner_only[
        [
            "GO_ID",
            "GO_term",
            "observed_count",
            "fold_enrichment_raw",
            "FDR"
        ]
    ].head(30)
)

# ------------------------------------------------
# 7. Save
# ------------------------------------------------

shared.to_csv(
    results_dir / "GO_BP_shared_RP_seeds_and_partners.tsv",
    sep="\t",
    index=False
)

rp_only.to_csv(
    results_dir / "GO_BP_RP_seed_specific.tsv",
    sep="\t",
    index=False
)

partner_only.to_csv(
    results_dir / "GO_BP_partner_specific.tsv",
    sep="\t",
    index=False
)

print("\n" + "=" * 70)
print("FILES SAVED")
print("=" * 70)

print("✓ GO_BP_shared_RP_seeds_and_partners.tsv")
print("✓ GO_BP_RP_seed_specific.tsv")
print("✓ GO_BP_partner_specific.tsv")

print("\nSTAGE 6.5 COMPLETE")

STAGE 6.5 — RP SEEDS vs RETINAL PARTNERS GO-BP COMPARISON

Checking input files:
PANTHER_GO_BP_RP_seeds_significant_FDR05.tsv: True
PANTHER_GO_BP_retinal_partners_significant_FDR05.tsv: True

RP significant terms      : 95
Partner significant terms : 2,105

GO TERM OVERLAP
RP GO terms with IDs       : 94
Partner GO terms with IDs  : 2,104
Shared significant terms   : 82
RP-only significant terms  : 12
Partner-only terms          : 2,022

TOP SHARED GO BIOLOGICAL PROCESSES


,GO_ID,GO_term,RP_observed,RP_expected,RP_fold,RP_FDR,partner_observed,partner_expected,partner_fold,partner_FDR
11,GO:0050953,sensory perception of light stimulus,33,0.81,40.99,8.640000e-41,76,27.39,2.77,1.920000e-17
12,GO:0007601,visual perception,31,0.77,40.09,5.500000e-38,72,26.31,2.74,3.880000e-16
4,GO:0001895,retina homeostasis,14,0.24,58.74,2.880000e-18,26,8.11,3.21,7.590000e-08
37,GO:0060271,cilium assembly,24,1.75,13.69,2.150000e-17,164,59.65,2.75,3.170000e-37
7,GO:0009583,detection of light stimulus,14,0.28,50.83,2.370000e-17,33,9.37,3.52,2.240000e-11
39,GO:0044782,cilium organization,24,1.94,12.35,2.070000e-16,186,66.14,2.81,3.510000e-44
3,GO:0045494,photoreceptor cell maintenance,12,0.19,62.94,5.060000e-16,21,6.49,3.24,1.680000e-06
45,GO:0120031,plasma membrane bounded cell projection assembly,24,2.12,11.33,1.190000e-15,185,72.09,2.57,7.080000e-37
46,GO:0030031,cell projection assembly,24,2.14,11.22,1.340000e-15,186,72.81,2.55,9.460000e-37
2,GO:0042461,photoreceptor cell development,11,0.16,67.00,4.870000e-15,14,5.59,2.51,4.590000e-03



TOP RP-SEED-SPECIFIC SIGNIFICANT PROCESSES


,GO_ID,GO_term,observed_count,fold_enrichment_raw,FDR
64,GO:0007600,sensory perception,35,7.80,1.720000e-18
69,GO:0050877,nervous system process,36,5.86,1.260000e-15
1,GO:0035845,photoreceptor cell outer segment organization,6,> 100,8.770000e-11
4,GO:0007603,"phototransduction, visible light",6,> 100,4.780000e-09
2,GO:0016056,G protein-coupled opsin signaling pathway,5,> 100,1.200000e-08
0,GO:0141190,transducin-mediated opsin signaling pathway,4,> 100,3.420000e-07
9,GO:0006776,vitamin A metabolic process,3,80.92,1.140000e-03
10,GO:0060296,regulation of cilium beat frequency involved i...,3,70.80,1.760000e-03
16,GO:0009642,response to light intensity,3,51.49,4.480000e-03
3,GO:1903546,protein localization to photoreceptor outer se...,2,> 100,1.360000e-02



TOP NETWORK-PARTNER-ONLY SIGNIFICANT PROCESSES


,GO_ID,GO_term,observed_count,fold_enrichment_raw,FDR
1790,GO:0008152,metabolic process,1998,1.57,3.990000e-156
1765,GO:0044238,primary metabolic process,1828,1.59,1.650000e-144
1466,GO:0009058,biosynthetic process,1220,1.85,4.860000e-135
1825,GO:0071840,cellular component organization or biogenesis,1523,1.53,1.280000e-94
302,GO:0008380,RNA splicing,222,3.99,9.020000e-94
261,GO:0000375,"RNA splicing, via transesterification reactions",200,4.17,2.690000e-90
263,GO:0000377,"RNA splicing, via transesterification reaction...",196,4.17,2.430000e-88
262,GO:0000398,"mRNA splicing, via spliceosome",196,4.17,2.650000e-88
1467,GO:0006139,nucleobase-containing compound metabolic process,848,1.85,3.980000e-86
430,GO:0006397,mRNA processing,233,3.58,1.160000e-83



FILES SAVED
✓ GO_BP_shared_RP_seeds_and_partners.tsv
✓ GO_BP_RP_seed_specific.tsv
✓ GO_BP_partner_specific.tsv

STAGE 6.5 COMPLETE
